# People Counting System on Google Colab
Run this notebook top-to-bottom to launch Kafka, the three services, and export sample results.

In [ ]:
!pip install ultralytics kafka-python pymongo fastapi uvicorn streamlit opencv-python-headless yt-dlp requests pillow -q


In [ ]:
%%bash
set -e
KAFKA_VERSION=3.7.2
KAFKA_SCALA=2.13
KAFKA_TGZ="kafka_${KAFKA_SCALA}-${KAFKA_VERSION}.tgz"
KAFKA_URL="https://downloads.apache.org/kafka/${KAFKA_VERSION}/${KAFKA_TGZ}"
rm -f "$KAFKA_TGZ"
rm -rf /opt/kafka
wget -nv "$KAFKA_URL" -O "$KAFKA_TGZ"
tar -xzf "$KAFKA_TGZ"
mv "kafka_${KAFKA_SCALA}-${KAFKA_VERSION}" /opt/kafka
KAFKA_HOME=/opt/kafka
export KAFKA_CLUSTER_ID=$($KAFKA_HOME/bin/kafka-storage.sh random-uuid)
$KAFKA_HOME/bin/kafka-storage.sh format -t $KAFKA_CLUSTER_ID -c $KAFKA_HOME/config/kraft/server.properties
$KAFKA_HOME/bin/kafka-server-start.sh $KAFKA_HOME/config/kraft/server.properties > /tmp/kafka.log 2>&1 &
sleep 10
$KAFKA_HOME/bin/kafka-topics.sh --create --topic raw-frames --bootstrap-server localhost:9092 --partitions 2 --replication-factor 1 || true
$KAFKA_HOME/bin/kafka-topics.sh --create --topic detection-results --bootstrap-server localhost:9092 --partitions 2 --replication-factor 1 || true


In [ ]:
!git clone https://github.com/YOUR_USERNAME/people-counting-system.git
%cd people-counting-system


In [ ]:
from utils.video_downloader import prepare_demo_video
video_path = prepare_demo_video()
print(video_path)


In [ ]:
import threading
import time
import uvicorn
from camera_server.producer import run_camera_server
from processing_server.consumer import run_processing_server
from storage_server.consumer import run_storage_consumer

def start_api():
    uvicorn.run('storage_server.api:app', host='0.0.0.0', port=8000, reload=False)

threads = [
    threading.Thread(target=run_storage_consumer, daemon=True),
    threading.Thread(target=run_processing_server, daemon=True),
    threading.Thread(target=start_api, daemon=True),
]
for t in threads:
    t.start()
time.sleep(5)
run_camera_server(video_path, max_frames=30)


In [ ]:
import json
import time
from storage_server.db import StorageDB

time.sleep(10)
db = StorageDB()
stats = db.get_stats()
latest = db.get_latest(10)
print(json.dumps(stats, indent=2))
print(json.dumps(latest[:2], indent=2))


In [ ]:
import json
from pathlib import Path
from utils.visualization import save_annotated_frame

Path('results').mkdir(exist_ok=True)
with open('results/sample_output.json', 'w') as f:
    json.dump(latest, f, indent=2)
sample = latest[0] if latest else None
if sample and sample.get('bounding_boxes'):
    from kafka import KafkaConsumer
    consumer = KafkaConsumer('raw-frames', bootstrap_servers='localhost:9092', value_deserializer=lambda m: json.loads(m.decode('utf-8')), auto_offset_reset='earliest', consumer_timeout_ms=5000)
    frame_message = next(iter(consumer), None)
    if frame_message is not None:
        save_annotated_frame(frame_message.value['frame_data'], sample['bounding_boxes'], 'results/annotated_frame.jpg')
print('Saved results/sample_output.json and results/annotated_frame.jpg')
